# 자동차인공지능 Project
# Subject - 차선 검출

### 본 문제는 차량의 전방 카메라 이미지로부터 차선을 검출하는 문제입니다.
### RGB 이미지를 입력으로 활용하여 **차선**을 이진 맵으로 검출하는 문제를 인공지능 모델을 활용하여 푸시오.
<br>

- [입력 데이터]
    - **차량 전방 카메라 이미지** 데이터로 구성
    - Train, test 데이터셋으로 분할되어 있음
- [정답값]
    - 2차원의 **차선 이진 맵**
    - 각 픽셀 별로 차선 영역=1, 차선이 아닌 영역=0으로 매핑되어 있음
    - Train 데이터에 대응하는 차선 이진 맵으로 구성
    
- 제공된 Train dataset을 자유롭게 본인만의 Train & Validation(검증) dataset으로 구성 가능
- **채점 평가 지표&emsp;&ensp;&nbsp;: F1 score**
- **인공지능 플랫폼&emsp;: Tensorflow**
- **환경&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;: Python, Numpy**

# Setup

### Import

In [46]:
# python --version 3.11.9
import os
import numpy as np              # version 1.26.4
import tensorflow as tf         # version 2.15.0
from ref.f1score import f1_score_tensorflow

In [47]:
X_train = np.load(os.path.join(os.getcwd(), 'ref', 'X_train.npz'))['arr_0']
y_train = np.load(os.path.join(os.getcwd(), 'ref', 'y_train.npz'))['arr_0']
X_test = np.load(os.path.join(os.getcwd(), 'ref', 'X_test.npz'))['arr_0']

### 예측 결과물 변수 y_pred 초기화

In [48]:
'''제출 시, X_test에 대한 모델의 예측 값을 y_pred 변수에 덮어 저장 필요'''
y_pred = np.zeros((len(X_test), y_train.shape[1], y_train.shape[2], 1)) # (N, H, W, 1)

print(f"y_pred shape: {y_pred.shape}")

y_pred shape: (1248, 128, 256, 1)


# ▼워크스페이스▼
* ipynb의 코드셀을 자유롭게 활용하여 인공지능 모델을 학습 후 아래 최종 제출물 코드를 활용하여 예측값을 저장하시오.

In [49]:
from tensorflow.keras.layers import Conv2D, MaxPooling2D, UpSampling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

model_cnn = Sequential([
    Conv2D(filters=16, kernel_size=(3, 3), strides=(1, 1), padding='same', activation='relu', input_shape=(128, 256, 3)),
    MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),
    Conv2D(filters=32, kernel_size=(3, 3), strides=(1, 1), padding='same', activation='relu'),
    MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),
    Conv2D(filters=64, kernel_size=(3, 3), strides=(1, 1), padding='same', activation='relu'),
    MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),
    UpSampling2D(size=2, interpolation='bilinear'),
    Conv2D(filters=32, kernel_size=(3, 3), strides=(1, 1), padding='same', activation='relu'),
    UpSampling2D(size=2, interpolation='bilinear'),
    Conv2D(filters=16, kernel_size=(3, 3), strides=(1, 1), padding='same', activation='relu'),
    UpSampling2D(size=2, interpolation='bilinear'),
    Conv2D(filters=1, kernel_size=(1, 1), strides=(1, 1), padding='valid', activation='sigmoid'),
])

X_train = X_train / 255.0
y_train = np.expand_dims(y_train, axis=-1)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    verbose=1,
    restore_best_weights=True
)

model_cnn.compile(optimizer=Adam(learning_rate=0.001),
              loss='binary_crossentropy',
              metrics=[f1_score_tensorflow])

history = model_cnn.fit(X_train, y_train, epochs=20, batch_size = 16, validation_split = 0.2, callbacks=[early_stop])

Epoch 1/20
258/258 [==============================] - 3s 8ms/step - loss: 0.1731 - f1_score_tensorflow: 3.2527e-04 - val_loss: 0.1377 - val_f1_score_tensorflow: 0.0000e+00
Epoch 2/20
258/258 [==============================] - 3s 10ms/step - loss: 0.1299 - f1_score_tensorflow: 0.0000e+00 - val_loss: 0.1231 - val_f1_score_tensorflow: 0.0000e+00
Epoch 3/20
258/258 [==============================] - 2s 7ms/step - loss: 0.1157 - f1_score_tensorflow: 0.1145 - val_loss: 0.1092 - val_f1_score_tensorflow: 0.1942
Epoch 4/20
258/258 [==============================] - 2s 9ms/step - loss: 0.1046 - f1_score_tensorflow: 0.2790 - val_loss: 0.1002 - val_f1_score_tensorflow: 0.2998
Epoch 5/20
258/258 [==============================] - 3s 11ms/step - loss: 0.0983 - f1_score_tensorflow: 0.3596 - val_loss: 0.0960 - val_f1_score_tensorflow: 0.4143
Epoch 6/20
258/258 [==============================] - 3s 10ms/step - loss: 0.0941 - f1_score_tensorflow: 0.4109 - val_loss: 0.0930 - val_f1_score_tensorflow: 0.45

In [50]:
X_test = X_test / 255.0
y_pred = model_cnn.predict(X_test)

39/39 [==============================] - 0s 5ms/step


- **제출 사항**
    - **해당 과제 .ipynb 파일**
    - **제출하는 .ipynb파일에 "submission.npy" 파일을 하나 이상 포함시킬 것**
    - **제출 파일은 train.ipynb로 할 것**

- **주의사항**
    - 모델에 X_test를 입력하여 추론할 때, shuffle 하지 말 것
    - 결과물을 여러 번 생성할 경우, 다음과 같은 형식으로 파일명을 관리한다  
    :submission1.npz, submission2.npz, submission3.npz, ...

In [51]:
print("반드시 출력물의 형태가 (1248, 128, 256, 1)인지 확인 필요.")
print(f"y_pred shape: {np.asarray(y_pred).shape}")

반드시 출력물의 형태가 (1248, 128, 256, 1)인지 확인 필요.
y_pred shape: (1248, 128, 256, 1)


In [54]:
np.savez(os.path.join(os.getcwd(), 'submission/submission1.npz'), y_pred)